# Solafune Satellite Precipitation Nowcasting — Kaggle Runner

Trains the Solafune nowcasting solution end-to-end on a Kaggle GPU (T4/P100/L4).

**Prerequisites**
1. Attach the `arnavchauhanra/solafune-dataset` Kaggle Dataset (Kaggle auto-extracts the ZIPs at upload; TIFs are already on the read-only `/kaggle/input` mount)
2. Enable **Internet** in notebook Settings (to `git clone` the repo)
3. Enable **GPU** (T4 / P100 / L4)

**Run order**
1. Setup — clone repo, autodetect paths on the `/kaggle/input` mount
2. Cache benchmark — Zarr vs memmap
3. Build train + eval Zarr caches (reads TIFs from `/kaggle/input`)
4. Compute normalization stats
5. Train (change `EXPERIMENT` to switch configs)
6. Inference + submission

**Session-safe** — hit the 12h wall? Re-run the training cell; `try_auto_resume()` picks up `last.pt`. `/kaggle/working` peak usage is ≈10 GB (Zarr cache only), well under the 20 GB quota.

## 1. Setup

In [ ]:
# --- Clone repo from GitHub ---
REPO_URL = 'https://github.com/arnav-chauhan-kgpian/solafune.git'
REPO_DIR = '/kaggle/working/solafune'

# Kaggle dataset mount (auto-extracted at upload — TIFs are directly readable)
DATASET_MOUNT = '/kaggle/input/solafune-dataset'
OUT_DIR = '/kaggle/working'

import os, sys, subprocess
from pathlib import Path

# ---- 1. Clone (or pull) the code repo -----------------------------------
if not Path(REPO_DIR).exists():
    print('cloning repo...')
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR])
else:
    print('repo present; pulling latest...')
    try:
        subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
    except subprocess.CalledProcessError:
        print('pull failed (probably no internet); continuing with cached copy')
sys.path.insert(0, REPO_DIR)

# ---- 2. Install missing pip packages ------------------------------------
for pkg in ('zarr', 'numcodecs', 'hydra-core', 'omegaconf'):
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# ---- 3. Autodetect the dataset roots on /kaggle/input --------------------
# Kaggle auto-extracted the ZIPs, so structure is:
#   /kaggle/input/solafune-dataset/solafune-train/train_dataset_<hash>/…
#   /kaggle/input/solafune-dataset/solafune-test/evaluation_dataset_<hash>/…
def _find(root, name_startswith):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f'{root} does not exist — attach the solafune-dataset dataset')
    for p in root.rglob('*'):
        if p.is_dir() and p.name.startswith(name_startswith):
            return p
    raise FileNotFoundError(f'no dir starting with {name_startswith!r} under {root}')

TRAIN_ROOT = _find(DATASET_MOUNT, 'train_dataset')
EVAL_ROOT  = _find(DATASET_MOUNT, 'evaluation_dataset')
TRAIN_CSV  = TRAIN_ROOT / 'train_dataset.csv'
EVAL_CSV   = EVAL_ROOT / 'evaluation_target.csv'
CACHE_ROOT = Path(OUT_DIR) / 'cache'
NORM_STATS = CACHE_ROOT / 'norm_stats.json'

print('REPO_DIR:  ', REPO_DIR)
print('TRAIN_ROOT:', TRAIN_ROOT)
print('EVAL_ROOT: ', EVAL_ROOT)
assert TRAIN_CSV.exists(), f'{TRAIN_CSV} missing'
assert EVAL_CSV.exists(),  f'{EVAL_CSV} missing'
print('setup complete.')

## 2. Cache Backend Benchmark

In [ ]:
import sys
if '/kaggle/working/solafune' not in sys.path:
    sys.path.insert(0, '/kaggle/working/solafune')

from src.data.cache.benchmark import run_benchmark
backend_json = CACHE_ROOT / 'backend.json'
if not backend_json.exists():
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    result = run_benchmark(output_path=backend_json, n_samples=200, channels=10, hw=81)
    print('recommended backend:', result.recommended)

import json
BACKEND = json.loads(backend_json.read_text())['recommended']
print('using backend:', BACKEND)

# On Kaggle we prefer Zarr for compression (disk-limited); switch here to override.
BACKEND = 'zarr'
print('locked backend to:', BACKEND)

## 3. Build Train + Eval Caches

In [ ]:
import sys
if '/kaggle/working/solafune' not in sys.path:
    sys.path.insert(0, '/kaggle/working/solafune')

import shutil
import pandas as pd
from src.data.cache import get_backend
from src.data.preprocessing import build_cache, build_cache_spec

# Compression settings (see cache size expectations below cell).
# zstd + clevel=7 is ~2x tighter than lz4 clevel=3 on satellite uint8 data
# and comfortably fits under Kaggle's 20 GB /kaggle/working quota.
COMPRESSOR = 'zstd'
CLEVEL     = 7

# If a *partial* cache from a failed run exists (spec.json missing but files
# present), wipe it so we start fresh. A cache whose spec.json exists is
# assumed complete and reused.
def _wipe_partial(out_dir):
    if not out_dir.exists():
        return
    if (out_dir / 'spec.json').exists():
        return
    print(f'wiping partial cache: {out_dir}')
    shutil.rmtree(out_dir, ignore_errors=True)

_wipe_partial(CACHE_ROOT / 'train')
_wipe_partial(CACHE_ROOT / 'eval')

def _build_cache_if_missing(csv, root, out_dir, load_gpm):
    spec_path = out_dir / 'spec.json'
    if spec_path.exists():
        print(f'cache exists: {out_dir}')
        return
    df = pd.read_csv(csv)
    spec, _ = build_cache_spec(df, out_dir, 'ir_only')
    cls = get_backend(BACKEND)
    if BACKEND == 'zarr':
        b = cls(spec, compressor=COMPRESSOR, clevel=CLEVEL)
    else:
        b = cls(spec)
    build_cache(csv, root, out_dir, b, 'ir_only', load_gpm=load_gpm, verbose_every=2000)
    b.close()
    print(f'cache built: {out_dir}')

_build_cache_if_missing(TRAIN_CSV, TRAIN_ROOT, CACHE_ROOT / 'train', load_gpm=True)
_build_cache_if_missing(EVAL_CSV,  EVAL_ROOT,  CACHE_ROOT / 'eval',  load_gpm=False)

# Disk report — should show plenty of headroom.
import subprocess
print('\n--- /kaggle/working disk usage ---')
subprocess.run(['df', '-h', '/kaggle/working'])
subprocess.run(['du', '-sh', str(CACHE_ROOT / 'train'), str(CACHE_ROOT / 'eval')])

## 4. Normalization Statistics

In [ ]:
import sys
if '/kaggle/working/solafune' not in sys.path:
    sys.path.insert(0, '/kaggle/working/solafune')

from src.constants import SATELLITES
from src.data.normalization import compute_norm_stats, save_norm_stats
from src.paths import sat_tif_path
from src.utils import parse_frame_list

if not NORM_STATS.exists():
    train_df = pd.read_csv(TRAIN_CSV)
    paths = {s: [] for s in SATELLITES}
    for _, row in train_df.iterrows():
        s = row['satellite_target']
        for f in parse_frame_list(row['last_30_minutes_observation_filename']):
            paths[s].append(sat_tif_path(TRAIN_ROOT, s, f))
    for s in SATELLITES:
        print(s, 'candidate TIFs:', len(paths[s]))
    stats = compute_norm_stats(paths, max_files_per_satellite=500, pixel_stride=2)
    save_norm_stats(NORM_STATS, stats)
print('norm stats:', NORM_STATS)

## 5. Sweep — Training + Inference (all runs in one pass)

Set `SWEEP` below to a list of `(EXPERIMENT, SEED, EPOCHS)` tuples. The cell trains, predicts, and writes a submission for each one, storing them under `submissions/<experiment>_seed<seed>/`. Section 8 then ensembles whatever's in there.

**Timing per run** (T4, batch=64, cosine scheduler):
- Cache + norm-stats already built → ~55 min per training run
- +5 min per inference pass
- **Budget: 60 min × N runs**. Session limit is 12h; a 3-run sweep fits comfortably.

Default sweep produces a 2-seed ensemble of the improved baseline. Uncomment the third row for a stronger 3-model ensemble with Conv3D temporal integration.

**No cell-by-cell interaction needed** — a single Save & Run All runs the whole sweep.

In [ ]:
import sys
if "/kaggle/working/solafune" not in sys.path:
    sys.path.insert(0, "/kaggle/working/solafune")

import gc, time, torch, shutil, pandas as pd
from pathlib import Path
from omegaconf import OmegaConf

from src.constants import max_active_channels
from src.data.dataloader import DataLoaderConfig, build_dataloader, build_sampler
from src.data.dataset import DatasetConfig, SolafuneDataset, split_indices_by_location
from src.experiment.tracker import snapshot_run
from src.models import build_model
from src.seed import seed_everything
from src.training.losses import build_loss
from src.training.schedulers import build_optimizer, build_scheduler
from src.training.trainer import Trainer, TrainerConfig
from src.training.ema import ExponentialMovingAverage
from src.inference.predict import PredictionConfig, predict
from src.inference.submission import write_submission

# ---------------------------------------------------------------------------
# SWEEP — list of (EXPERIMENT, SEED, EPOCHS). Each row = one full training +
# inference run. Every run writes its submission to
# /kaggle/working/submissions/<experiment>_seed<seed>/.
# ---------------------------------------------------------------------------
SWEEP = [
    ("exp0_baseline", 42, 30),
    ("exp0_baseline", 43, 30),
    # ("exp2_conv3d",           42, 30),   # uncomment for a 3-model ensemble
    # ("exp2_conv3d_ensemble",  42, 30),
    # ("exp6_efficientnet",     42, 30),
]

# ---------------------------------------------------------------------------
# TRAINING PROFILE — baked in for every sweep run
# ---------------------------------------------------------------------------
TRAINING_OVERRIDES = [
    "training.ema_enabled=false",
    "training.early_stop_patience=12",
    "data.batch_size=64",
    "scheduler=cosine",
    "scheduler.warmup_epochs=2",
    "optimizer.lr=1.5e-3",
    "loss.rain_weight_scale=1.5",
    "loss.mae_weight=0.3",
]

# ---------------------------------------------------------------------------
# Config composition
# ---------------------------------------------------------------------------
from hydra import compose, initialize_config_dir
import hydra
HYDRA_DIR = str(Path(REPO_DIR) / "configs")


def _make_cfg(experiment, seed, epochs, training_dir):
    if hydra.core.global_hydra.GlobalHydra.instance().is_initialized():
        hydra.core.global_hydra.GlobalHydra.instance().clear()
    overrides = [
        f"data.cache_dir={CACHE_ROOT}/train",
        f"data.norm_stats_path={NORM_STATS}",
        f"data.train_csv={TRAIN_CSV}",
        f"data.eval_csv={EVAL_CSV}",
        f"data.train_root={CACHE_ROOT}/train",
        f"data.eval_root={CACHE_ROOT}/eval",
        f"data.cache.backend={BACKEND}",
        f"training.output_dir={training_dir}",
        f"training.epochs={epochs}",
        f"seed={seed}",
        f"experiment={experiment}",
    ] + TRAINING_OVERRIDES
    with initialize_config_dir(config_dir=HYDRA_DIR, version_base=None):
        return compose(config_name="config", overrides=overrides)


def _run_one(experiment, seed, epochs):
    print(f"\n{'=' * 70}\n=== TRAINING {experiment} seed={seed} epochs={epochs}\n{'=' * 70}")
    t_start = time.perf_counter()

    training_dir = Path(OUT_DIR) / "training" / experiment / f"seed_{seed}"
    training_dir.mkdir(parents=True, exist_ok=True)
    cfg = _make_cfg(experiment, seed, epochs, training_dir)

    seed_everything(int(cfg.seed))
    snapshot_run(training_dir, cfg, repo_root=Path(REPO_DIR))

    # ----- data -----
    ds_cfg = DatasetConfig(
        cache_dir=Path(cfg.data.cache_dir),
        csv_path=Path(cfg.data.train_csv),
        norm_stats_path=Path(cfg.data.norm_stats_path),
        image_size=int(cfg.data.image_size),
        bands=str(cfg.data.bands),
        include_diff_frames=bool(cfg.data.include_diff_frames),
        missing_frame_strategy=str(cfg.data.missing_frame_strategy),
        cache_backend=BACKEND,
    )
    df = pd.read_csv(ds_cfg.csv_path)
    train_idx, val_idx = split_indices_by_location(df, list(cfg.data.val_locations))
    train_ds = SolafuneDataset(ds_cfg, df=df, indices=train_idx)
    val_ds = SolafuneDataset(ds_cfg, df=df, indices=val_idx)
    print(f"    train: {len(train_ds)}   val: {len(val_ds)}")

    dl = DataLoaderConfig(
        batch_size=int(cfg.data.batch_size), num_workers=int(cfg.data.num_workers),
        pin_memory=True, persistent_workers=True, prefetch_factor=2, drop_last=True,
    )
    train_loader = build_dataloader(
        train_ds, dl,
        sampler=build_sampler(train_ds, str(cfg.data.sampling.strategy),
                              precip_weight_scale=float(cfg.data.sampling.precip_weight_scale)),
        base_seed=int(cfg.seed),
    )
    val_loader = build_dataloader(val_ds, dl, shuffle=False, base_seed=int(cfg.seed))

    # ----- model -----
    mcfg = OmegaConf.to_container(cfg.model, resolve=True)
    mcfg["in_channels_per_frame"] = max_active_channels(str(cfg.data.bands))
    mcfg["n_frames"] = int(cfg.data.frames)
    mcfg["n_diff_frames"] = int(cfg.data.frames - 1) if cfg.data.include_diff_frames else 0
    model = build_model(mcfg)
    print(f"    params: {sum(p.numel() for p in model.parameters()):,}")

    # ----- training -----
    loss_fn = build_loss(OmegaConf.to_container(cfg.loss))
    opt = build_optimizer(model, OmegaConf.to_container(cfg.optimizer))
    sched, seb = build_scheduler(
        opt, OmegaConf.to_container(cfg.scheduler),
        steps_per_epoch=len(train_loader), epochs=int(cfg.training.epochs),
    )
    tcfg = TrainerConfig(**OmegaConf.to_container(cfg.training))
    tcfg.step_scheduler_each_batch = seb
    trainer = Trainer(model, opt, sched, loss_fn, train_loader, val_loader, tcfg)
    trainer.try_auto_resume()
    best_val = trainer.fit()
    print(f"    best val metric: {best_val}")

    # ----- load best.pt for inference -----
    best_ckpt = training_dir / "checkpoints" / "best.pt"
    state = torch.load(best_ckpt, map_location="cpu", weights_only=False)
    weight_source = state.get("best_source", "raw")
    if weight_source == "ema" and state.get("ema") is not None:
        ema = ExponentialMovingAverage(model, decay=cfg.training.ema_decay)
        ema.load_state_dict(state["ema"])
        ema.apply(model).__enter__()
        print("    loaded EMA weights")
    else:
        model.load_state_dict(state["model"])
        print("    loaded RAW weights")

    # ----- inference -----
    eval_ds_cfg = DatasetConfig(
        cache_dir=CACHE_ROOT / "eval",
        csv_path=EVAL_CSV,
        norm_stats_path=NORM_STATS,
        image_size=int(cfg.data.image_size), bands="ir_only",
        include_diff_frames=True, cache_backend=BACKEND,
    )
    eval_ds = SolafuneDataset(eval_ds_cfg)
    eval_loader = build_dataloader(
        eval_ds,
        DataLoaderConfig(batch_size=32, num_workers=2, pin_memory=True,
                         persistent_workers=True, prefetch_factor=2, drop_last=False),
        shuffle=False, base_seed=int(cfg.seed),
    )
    preds = predict(model, eval_loader,
                    PredictionConfig(amp=True, tta=True, rain_mask_threshold=0.15))
    print(f"    predictions: shape={preds.shape} min/max={preds.min():.3f}/{preds.max():.3f}"
          f" mean={preds.mean():.4f} nonzero%={(preds > 0).mean() * 100:.1f}%")

    # ----- submission -----
    sub_root = Path(OUT_DIR) / "submissions" / f"{experiment}_seed{seed}"
    test_files = sub_root / "test_files"
    n_written = write_submission(preds, EVAL_CSV, test_files)
    shutil.copy(EVAL_CSV, sub_root / "evaluation_target.csv")
    archive = shutil.make_archive(str(Path(OUT_DIR) / f"submission_{experiment}_seed{seed}"),
                                    "zip", str(sub_root))
    # stable-name copy for the audit cell
    stable_sub = Path(OUT_DIR) / "submission"
    if stable_sub.exists():
        shutil.rmtree(stable_sub)
    shutil.copytree(sub_root, stable_sub)

    dt = time.perf_counter() - t_start
    print(f"    files: {n_written}   archive: {archive} ({Path(archive).stat().st_size / 1e6:.1f} MB)")
    print(f"    elapsed: {dt:.0f}s ({dt / 60:.1f} min)")

    # free memory before next run
    del trainer, model, opt, sched, train_loader, val_loader, eval_loader, preds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return best_val


results = []
for experiment, seed, epochs in SWEEP:
    # ----- skip rows that already produced a complete submission -----
    sub_root   = Path(OUT_DIR) / "submissions" / f"{experiment}_seed{seed}"
    test_files = sub_root / "test_files"
    if test_files.is_dir():
        n = sum(1 for _ in test_files.iterdir())
        if n >= 29000:   # eval set is 29090 files; allow a small margin
            print(f"=== SKIP {experiment} seed={seed} (submission already has {n} files)")
            results.append((experiment, seed, "skipped"))
            # keep stable-name copy pointed at this submission
            stable_sub = Path(OUT_DIR) / "submission"
            if stable_sub.exists():
                shutil.rmtree(stable_sub)
            shutil.copytree(sub_root, stable_sub)
            continue
    val = _run_one(experiment, seed, epochs)
    results.append((experiment, seed, val))

print(f"\n\n{'=' * 70}\n=== SWEEP COMPLETE ===\n{'=' * 70}")
for exp, seed, val in results:
    print(f"  {exp:<25} seed={seed}   best_val_metric={val}")


## 6. Diagnostic plots

In [ ]:
import sys
if "/kaggle/working/solafune" not in sys.path:
    sys.path.insert(0, "/kaggle/working/solafune")

# Plot the LAST run in the SWEEP.
from src.visualization import plot_training_curves, plot_val_curves
last_exp, last_seed, _ = SWEEP[-1]
last_dir = Path(OUT_DIR) / "training" / last_exp / f"seed_{last_seed}"
plot_training_curves(last_dir / "train_metrics.csv", last_dir / "plots" / "train.png")
plot_val_curves(last_dir / "val_metrics.csv", last_dir / "plots" / "val.png")
print(f"plots for {last_exp} seed={last_seed} saved in {last_dir}/plots")


## 7. Submission audit (last run in sweep)

The sweep cell above already wrote a submission zip per run. This cell only audits format on the LAST run's files. The stable path `/kaggle/working/submission/` always points to the last completed run.

For per-run submission zips, look at `/kaggle/working/submission_<experiment>_seed<seed>.zip`.

In [ ]:
import sys
if '/kaggle/working/solafune' not in sys.path:
    sys.path.insert(0, '/kaggle/working/solafune')

# Final format audit — reads a random subset back and asserts the Solafune contract
import random, numpy as np
from src.utils.io import read_gpm_tif
random.seed(0)
sample_names = random.sample(list(pd.read_csv(EVAL_CSV)['gpm_imerg_filename']), 25)
for name in sample_names:
    arr, meta = read_gpm_tif(TEST_FILES / name)
    assert arr.shape == (41, 41), (name, arr.shape)
    assert arr.dtype == np.float32, (name, arr.dtype)
    assert np.isfinite(arr).all(), name
print(f'audit passed: 25/25 sampled TIFs conform to (41,41) float32 finite')
print('submission ready at:', SUB_DIR)

## 8. Ensemble submissions

Averages predictions from all runs stored under `submissions/<exp>_seed<n>/`. Every subdirectory of `submissions/` gets equal weight by default; set the `WEIGHTS` dict below to override per-run.

Delete any run subdirectory before running this cell to exclude it from the ensemble.

In [ ]:
import sys
if '/kaggle/working/solafune' not in sys.path:
    sys.path.insert(0, '/kaggle/working/solafune')

import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from src.utils.io import read_gpm_tif, write_gpm_tif

# --- Config ---------------------------------------------------------------
SUBMISSIONS_ROOT = Path(OUT_DIR) / 'submissions'
ENSEMBLE_TAG     = 'ensemble'
# Explicit weight per submission dir name; missing entries default to 1.0.
# Example: {'exp0_baseline_seed42': 1.0, 'exp2_conv3d_seed42': 2.0}
WEIGHTS: dict = {}
# Post-averaging rain-mask threshold — same as single-model default.
# Higher value zeros out more low-confidence pixels (more precision, less recall).
POST_RAIN_MASK_THRESHOLD = 0.15
# Optional: clip to [0, MAX_MM_H] to guard against runaway predictions.
MAX_MM_H = 60.0

# --- Discover runs --------------------------------------------------------
run_dirs = sorted(p for p in SUBMISSIONS_ROOT.iterdir()
                  if p.is_dir() and (p / 'test_files').is_dir())
if len(run_dirs) < 2:
    raise RuntimeError(
        f'need at least 2 runs to ensemble, found {len(run_dirs)} in {SUBMISSIONS_ROOT}. '
        f'Change SEED and rerun cells 10-17 to add another run.'
    )
weights = np.array([float(WEIGHTS.get(p.name, 1.0)) for p in run_dirs], dtype=np.float64)
weights /= weights.sum()
for p, w in zip(run_dirs, weights):
    print(f'  {p.name}   weight={w:.3f}')

# --- Verify all runs share the same file list ----------------------------
first_files = sorted((run_dirs[0] / 'test_files').iterdir())
first_names = {p.name for p in first_files}
for p in run_dirs[1:]:
    other = {f.name for f in (p / 'test_files').iterdir()}
    missing = first_names - other
    extra = other - first_names
    if missing or extra:
        raise RuntimeError(f'{p.name} filelist mismatch: missing={len(missing)} extra={len(extra)}')
print(f'\nall {len(run_dirs)} runs cover {len(first_names)} files')

# --- Average ---------------------------------------------------------------
ENS_ROOT   = SUBMISSIONS_ROOT / ENSEMBLE_TAG
ENS_FILES  = ENS_ROOT / 'test_files'
if ENS_ROOT.exists():
    shutil.rmtree(ENS_ROOT)
ENS_FILES.mkdir(parents=True, exist_ok=True)

names = sorted(first_names)
n = len(names)
print(f'\naveraging {n} files across {len(run_dirs)} runs...')
for i, name in enumerate(names, 1):
    acc = np.zeros((41, 41), dtype=np.float64)
    for run_dir, w in zip(run_dirs, weights):
        arr, _ = read_gpm_tif(run_dir / 'test_files' / name)
        acc += w * arr.astype(np.float64)
    acc = np.clip(acc, 0.0, MAX_MM_H).astype(np.float32)
    # Apply post-ensemble rain mask (any pixel < threshold → 0) to reduce
    # false-positive tiny predictions.
    acc[acc < POST_RAIN_MASK_THRESHOLD] = 0.0
    write_gpm_tif(ENS_FILES / name, acc)
    if i % 5000 == 0 or i == n:
        print(f'  [{i}/{n}] {i * 100 // n}%', flush=True)

# --- Copy CSV + zip -------------------------------------------------------
shutil.copy(EVAL_CSV, ENS_ROOT / 'evaluation_target.csv')
ens_archive = shutil.make_archive(
    str(Path(OUT_DIR) / 'submission_ensemble'), 'zip', str(ENS_ROOT),
)
print(f'\nensemble archive: {ens_archive}   size: {Path(ens_archive).stat().st_size / 1e6:.1f} MB')

# --- Quick sanity checks --------------------------------------------------
sample_names = list(pd.read_csv(EVAL_CSV)['gpm_imerg_filename'].sample(25, random_state=0))
for name in sample_names:
    arr, _ = read_gpm_tif(ENS_FILES / name)
    assert arr.shape == (41, 41), name
    assert arr.dtype == np.float32, name
    assert np.isfinite(arr).all(), name
print('audit: 25/25 sampled TIFs conform to (41,41) float32 finite')

# Prediction statistics comparison
def _stats(dir_):
    p = list((dir_ / 'test_files').iterdir())[:500]
    vals = np.concatenate([read_gpm_tif(f)[0].ravel() for f in p])
    return {'mean': float(vals.mean()), 'nonzero%': float((vals > 0).mean() * 100),
            'p95': float(np.percentile(vals, 95)), 'max': float(vals.max())}
print('\nprediction statistics (over 500-file sample):')
print(f'  {"run":<40s} {"mean":>8s} {"nz%":>6s} {"p95":>8s} {"max":>8s}')
for p in run_dirs + [ENS_ROOT]:
    s = _stats(p)
    print(f'  {p.name:<40s} {s["mean"]:>8.4f} {s["nonzero%"]:>5.1f}% {s["p95"]:>8.3f} {s["max"]:>8.3f}')

## 9. Iteration workflow — one Save & Run All

Everything above runs top-to-bottom in a single Save & Run All pass:

1. Cell 2: clone repo, wire paths
2. Cell 4: cache backend benchmark (~30 s, cached after first run)
3. Cell 6: build train + eval Zarr caches (~2 h, cached after first run)
4. Cell 8: norm stats (~2 min, cached after first run)
5. **Cell 10 (sweep):** trains + predicts every `(EXPERIMENT, SEED)` in `SWEEP`. ~60 min per row.
6. Plots + audit cells reference the LAST sweep run automatically.
7. Cell 20 (ensemble): averages every subdirectory under `submissions/`, writes `submission_ensemble.zip`.

**To change what runs**: edit the `SWEEP` list at the top of cell 10. Comment / uncomment rows. Nothing else needs touching.